## 📎 Code this notebook uses
*(how to run, and which code each step uses: [`README.md`](README.md))*

- **`code/pipeline/compute/`** (in this order): `compute_annual_wave.py`, `compute_own_diffusivity_fit.py`,
  `compute_annual_wave_surface_test.py`, `compute_transient_warming.py`, `compute_joint_mcmc.py`
- **`code/pipeline/figures/`:** `make_joint_figures.py` (Figures 5, 6)

# 05 — The annual wave, the warming after deployment, and the posterior

Three independent checks on the result: the yearly temperature wave measured
in the full record and modeled (Section 3.4, Figure 5, SI Table S12); the
warming of the sensors after the astronauts disturbed the surface (Section
4.3, SI Table S7); and the posterior with the basal heat flux and the density
also free (Section 4.1, Figure 6, SI Table S8).

In [ ]:
# How much to run (README.md explains the three modes):
#   "show"  : run nothing, read the stored results            (minutes)
#   "quick" : run the short steps and the fast form of long ones  (~30 min for all notebooks)
#   "full"  : run everything from scratch, as for the paper    (about 6 h for all notebooks)
MODE = "show"

import pandas as pd
from lunar import notebook as nb
nb.set_mode(MODE)
SITES_ = ("A15", "A17")

## 1. The annual wave (Section 3.4, SI Table S12)

The yearly wave measured in the full record, compared with the model forced by
the real Sun–Moon geometry. Full run about 3 min.

In [ ]:
nb.step("compute/compute_annual_wave.py", minutes=3, writes="annual_wave.json")
aw = nb.load("annual_wave.json")["sites"]
display(pd.DataFrame([(s, round(aw[s]["variants"]["nominal"]["kd_mW"], 2),
                       " – ".join(f"{x:.2f}" for x in aw[s]["variants"]["nominal"]["ci95"])) for s in SITES_],
                     columns=["site", "K_d from the annual wave", "95% interval"]))
chk = nb.Checks("Compared with the paper (Section 3.4):")
for s, (kd, lo, hi) in (("A15", ("5.7", "5.3", "6.1")), ("A17", ("5.6", "5.2", "6.1"))):
    n = aw[s]["variants"]["nominal"]
    chk.add(f"{s}: K_d", n["kd_mW"], kd, ".1f")
    chk.add(f"{s}: 95% lower", n["ci95"][0], lo, ".1f")
    chk.add(f"{s}: 95% upper", n["ci95"][1], hi, ".1f")
chk.show()

## 2. The joint fit with our own diffusivities (SI Table S12, lower block)

Langseth et al. (1976)'s diffusivities replaced by ones measured here from
the annual wave's amplitude, its delay, or both.

In [ ]:
nb.step("compute/compute_own_diffusivity_fit.py", minutes=0.3, writes="own_diffusivity_fit.json")
od = nb.load("own_diffusivity_fit.json")["variants"]
display(pd.DataFrame({v: {s: round(od[v][s]["kd_star_mW"], 2) for s in SITES_} for v in od}).T
        .rename_axis("diffusivity used"))
own = [v for v in od if v != "langseth1976"]
chk = nb.Checks("Compared with the paper (Section 3.4):")
for s, rng in (("A15", "4.9-6.1"), ("A17", "5.8-6.1")):
    ks = [od[v][s]["kd_star_mW"] for v in own]
    chk.add_text(f"{s}: K_d range with our own diffusivities", f"{min(ks):.1f}-{max(ks):.1f}", rng)
chk.show()

## 3. A different near-surface layer against the annual wave (Section 4.4, SI Table S12)

In [ ]:
nb.step("compute/compute_annual_wave_surface_test.py", minutes=6, writes="annual_wave_surface_test.json")
st = nb.load("annual_wave_surface_test.json")["sites"]
display(pd.DataFrame({s: {c: round(st[s][c]["kd_mW"], 2) for c in st[s]} for s in SITES_})
        .rename_axis("near-surface layer"))

## 4. The warming after deployment (Section 4.3, SI Table S7)

A darkening of the surface at deployment, of the size that reproduces the
measured warming rates, modeled and removed from each window; then the fit
redone.

In [ ]:
nb.step("compute/compute_transient_warming.py", minutes=2, writes="transient_warming.json")
tw = nb.load("transient_warming.json")["sites"]
display(pd.DataFrame({s: {c: round(v["surface_as_measured"]["d_kd_mW"], 3) for c, v in tw[s]["cases"].items()}
                      for s in SITES_}).rename_axis("surface step: change of K_d*"))
chk = nb.Checks("Compared with the paper (Section 4.3):")
chk.add("A17, step that reproduces the warming (1.6 K)", -tw["A17"]["cases"]["step_1.6K"]["surface_as_measured"]["d_kd_mW"], "0.06", ".2f")
chk.add("A17, step about twice as large (3.5 K)", -tw["A17"]["cases"]["step_3.5K"]["surface_as_measured"]["d_kd_mW"], "0.45", ".2f")
chk.add("A15, largest change", max(abs(v["surface_as_measured"]["d_kd_mW"]) for v in tw["A15"]["cases"].values()), "<0.05")
chk.show()

## 5. The posterior (Section 4.1, Figure 6, SI Table S8)

The basal heat flux and the deep density sampled together with the albedo and
$K_d$. The full run solves a grid of models first (about 2 h, resumable);
"quick" samples the stored grid (`joint_mcmc_grid.npz`) in about a minute.

In [ ]:
nb.step("compute/compute_joint_mcmc.py", minutes=120, writes="joint_mcmc.json",
        quick=("--sample",), quick_needs="results/joint_mcmc_grid.npz")
mc = nb.load("joint_mcmc.json")
display(pd.DataFrame({s: {k: f'{v["p50"]:.3g} [{v["p2.5"]:.3g}, {v["p97.5"]:.3g}]' for k, v in mc[s]["posterior"].items()}
                      for s in SITES_}).rename_axis("median [95%]"))
chk = nb.Checks("Compared with the paper (Section 4.1):")
for s, (lo, hi) in (("A15", ("4.1", "5.7")), ("A17", ("5.1", "6.9"))):
    k = mc[s]["posterior"]["kd_mW"]
    chk.add(f"{s}: K_d 95% lower", k["p2.5"], lo, ".1f")
    chk.add(f"{s}: K_d 95% upper", k["p97.5"], hi, ".1f")
c = mc["contrast_mW"]
chk.add("A17 - A15, median", c["p50"], "+1.1", "+.1f")
chk.add("A17 - A15, 95% lower", c["p2.5"], "-0.1", "+.1f")
chk.add("A17 - A15, 95% upper", c["p97.5"], "+2.3", "+.1f")
chk.show()

## 6. Figures

In [ ]:
nb.figures("figures/make_joint_figures.py", makes=("fig_joint_annual_wave", "fig_joint_mcmc"), minutes=2)

**Figure 5.** The annual wave in the full record against the forward model.

In [ ]:
nb.show("fig_joint_annual_wave")

**Figure 6.** The posterior distributions.

In [ ]:
nb.show("fig_joint_mcmc")

## 7. Every figure of the paper

In [ ]:
paper_figs = {"Figure 1": "fig_joint_intro_column", "Figure 2": "fig_joint_apollo_timeline_probes",
              "Figure 3": "fig_joint_thermal_profiles", "Figure 4": "fig_joint_constraints",
              "Figure 5": "fig_joint_annual_wave", "Figure 6": "fig_joint_mcmc",
              "SI Figure S1": "fig_context_map", "SI Figure S2": "fig_study_overview",
              "SI Figure S3": "fig_joint_anchor_process", "SI Figure S4": "fig_joint_amplitude_vs_depth",
              "SI Figure S5": "fig_joint_diviner_closure", "SI Figure S6": "fig_joint_robustness"}
for k, f in paper_figs.items():
    print(f"  {'✓' if (nb.FIGURES / f'{f}.pdf').exists() else '✗ not made'}  {k:13s} figures/{f}.pdf")

That is the whole paper. **06_performance.ipynb** is optional: the solver
in Python against an independent C++ copy.